# 01 — Feature Engineering
## Stationarity, Differencing, Lags & Rolling Statistics

### Purpose
Test stationarity of all indicators using ADF and KPSS tests.
First-difference non-stationary columns. Build lag features (t-1).
Compute rolling mean and standard deviation (3yr, 5yr).

### Input
- `../data/01_panel_cleaned.csv`

### Output
- `../data/02_panel_features.csv`

### Run after → Run before
`00_data_preparation.ipynb` → `02_instability_index_eda.ipynb`

In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")



TRAIN_END = 2019
OBSERVED_END = 2023
SCENARIO_START = 2024
SCENARIO_END = 2026

df_panel = pd.read_csv(r"data/01_panel_cleaned.csv")
df_panel = df_panel.sort_values(["COUNTRY", "YEAR"]).copy()

df_panel["Data_Split"] = np.select(
    [
        df_panel["YEAR"] <= TRAIN_END,
        df_panel["YEAR"].between(2020, OBSERVED_END),
        df_panel["YEAR"].between(SCENARIO_START, SCENARIO_END),
    ],
    [
        "train_observed",
        "test_observed",
        "scenario_projection",
    ],
    default="outside_scope",
)

print(df_panel["Data_Split"].value_counts())
print(f"Loaded: {df_panel.shape} | Countries: {df_panel['COUNTRY'].nunique()}")


Data_Split
train_observed         4302
test_observed           692
scenario_projection     513
Name: count, dtype: int64
Loaded: (5507, 24) | Countries: 175


In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df_panel = pd.read_csv("data/01_panel_cleaned.csv")
print(f"Loaded: {df_panel.shape} | Countries: {df_panel['COUNTRY'].nunique()}")
df_panel.head()

Loaded: (5507, 23) | Countries: 175


,COUNTRY,YEAR,Inflation,Current_Account,Expenditure,Exports,Investment,Debt,GDP_Growth,Savings,...,Inflation_imputed,Current_Account_imputed,Expenditure_imputed,Exports_imputed,Imports_imputed,Savings_imputed,Investment_imputed,Debt_imputed,Fiscal_Balance_imputed,Revenue_imputed
0,"Afghanistan, Islamic Republic of",2003,35.663,29.616,11.927,49.575,30.102,270.602,8.692,59.718,...,False,False,False,False,False,False,False,False,False,False
1,"Afghanistan, Islamic Republic of",2004,16.358,37.216,15.069,-8.863,35.354,244.967,0.671,72.570,...,False,False,False,False,False,False,False,False,False,False
2,"Afghanistan, Islamic Republic of",2005,10.569,30.226,15.651,40.082,37.048,206.356,11.830,67.274,...,False,False,False,False,False,False,False,False,False,False
3,"Afghanistan, Islamic Republic of",2006,6.785,20.844,18.262,-5.943,29.489,22.985,5.361,50.333,...,False,False,False,False,False,False,False,False,False,False
4,"Afghanistan, Islamic Republic of",2007,8.681,63.390,21.448,-12.758,55.852,20.137,13.340,119.243,...,False,False,False,False,False,False,False,False,False,False


In [8]:


warnings.filterwarnings("ignore")

from statsmodels.tsa.stattools import adfuller, kpss

cols_to_test = [
    "GDP_Growth", "Inflation", "Debt", "Fiscal_Balance",
    "Current_Account", "Exports", "Imports",
    "Revenue", "Expenditure", "Savings", "Investment"
]

stationarity_sample = df_panel[df_panel["YEAR"] <= TRAIN_END].copy()

results = []

for col in cols_to_test:
    adf_pvals = []
    kpss_pvals = []

    for country, grp in stationarity_sample.groupby("COUNTRY"):
        series = grp[col].dropna()

        if len(series) < 8:
            continue

        try:
            adf_pvals.append(adfuller(series, autolag="AIC")[1])
        except Exception:
            pass

        try:
            kpss_pvals.append(kpss(series, regression="c", nlags="auto")[1])
        except Exception:
            pass

    results.append({
        "Column": col,
        "ADF_pct_reject": np.mean(np.array(adf_pvals) < 0.05) * 100,
        "KPSS_pct_reject": np.mean(np.array(kpss_pvals) < 0.05) * 100,
        "ADF_median_p": np.median(adf_pvals),
        "KPSS_median_p": np.median(kpss_pvals),
    })

stationarity_df = pd.DataFrame(results)

stationarity_df["Suggested_Difference"] = (
    (stationarity_df["ADF_pct_reject"] < 50)
    | (stationarity_df["KPSS_pct_reject"] > 40)
)

display(stationarity_df)



non_stationary_cols = stationarity_df.loc[
    stationarity_df["Suggested_Difference"],
    "Column"
].tolist()

# Keep GDP_Growth as target level, not differenced target
non_stationary_cols = [
    col for col in non_stationary_cols
    if col != "GDP_Growth"
]

print("Columns to difference based on training years only:")
print(non_stationary_cols)

,Column,ADF_pct_reject,KPSS_pct_reject,ADF_median_p,KPSS_median_p,Suggested_Difference
0,GDP_Growth,58.285714,13.714286,0.023832,0.100000,False
1,Inflation,53.142857,33.714286,0.037840,0.085165,False
2,Debt,24.000000,38.857143,0.410394,0.075172,True
3,Fiscal_Balance,34.857143,15.428571,0.120710,0.100000,True
4,Current_Account,28.571429,20.000000,0.194698,0.100000,True
5,Exports,77.142857,14.285714,0.002014,0.100000,False
6,Imports,72.000000,4.571429,0.002311,0.100000,False
7,Revenue,34.285714,37.142857,0.186501,0.082464,True
8,Expenditure,21.714286,38.857143,0.266224,0.085455,True
9,Savings,23.428571,33.714286,0.188138,0.092927,True


Columns to difference based on training years only:
['Debt', 'Fiscal_Balance', 'Current_Account', 'Revenue', 'Expenditure', 'Savings', 'Investment']


In [10]:

non_stationary_cols = stationarity_df.loc[
    stationarity_df["Suggested_Difference"],
    "Column"
].tolist()

# Keep GDP_Growth as target level, not differenced target
non_stationary_cols = [
    col for col in non_stationary_cols
    if col != "GDP_Growth"
]

print("Columns to difference based on training years only:")
print(non_stationary_cols)
# Verify stationarity after differencing
for col in non_stationary_cols:
    pvals = []
    for country, grp in stationarity_sample.groupby("COUNTRY"):
        s = grp[col].diff().dropna()
        if len(s) >= 8:
            try:
                pvals.append(adfuller(s, autolag="AIC")[1])
            except Exception:
                pass
    pct = np.mean(np.array(pvals) < 0.05) * 100
    print(f"{col}_diff → {pct:.1f}% stationary after differencing")

Columns to difference based on training years only:
['Debt', 'Fiscal_Balance', 'Current_Account', 'Revenue', 'Expenditure', 'Savings', 'Investment']
Debt_diff → 47.7% stationary after differencing
Fiscal_Balance_diff → 70.1% stationary after differencing
Current_Account_diff → 66.7% stationary after differencing
Revenue_diff → 62.6% stationary after differencing
Expenditure_diff → 70.7% stationary after differencing
Savings_diff → 62.1% stationary after differencing
Investment_diff → 81.0% stationary after differencing


In [11]:
for col in non_stationary_cols:
    df_panel[f"{col}_diff"] = (
        df_panel.groupby("COUNTRY")[col]
        .transform(lambda x: x.diff())
    )

roll_cols = [
    "Inflation", "GDP_Growth", "Exports", "Imports",
    "Fiscal_Balance", "Current_Account",
    "Debt", "Expenditure", "Revenue", "Savings", "Investment"
]

for col in roll_cols:
    for window in [3, 5]:
        df_panel[f"{col}_rollmean{window}"] = (
            df_panel.groupby("COUNTRY")[col]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=2).mean())
        )

        df_panel[f"{col}_rollstd{window}"] = (
            df_panel.groupby("COUNTRY")[col]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=2).std())
        )


In [12]:
stationary_cols = [
    "GDP_Growth",
    "Inflation",
    "Exports",
    "Imports",
    "Fiscal_Balance",
    "Current_Account",
]

differenced_cols = [f"{col}_diff" for col in non_stationary_cols]

all_feature_cols = stationary_cols + differenced_cols

for col in all_feature_cols:
    df_panel[f"{col}_lag1"] = (
        df_panel.groupby("COUNTRY")[col]
        .transform(lambda x: x.shift(1))
    )

df_panel["GDP_Growth_rollmean3"] = (
    df_panel.groupby("COUNTRY")["GDP_Growth"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=2).mean())
)

df_panel["Inflation_lag1_log"] = (
    np.sign(df_panel["Inflation_lag1"])
    * np.log1p(np.abs(df_panel["Inflation_lag1"]))
)

In [ ]:
# # ============================================================
# # Additional volatility features
# # Use lagged rolling volatility only
# # ============================================================

# volatility_base_cols = [
#     "GDP_Growth",
#     "Inflation",
#     # "Exports",
#     "Imports",
#     "Fiscal_Balance",
#     "Current_Account",
#     "Debt",
#     "Revenue",
#     "Expenditure",
#     "Savings",
#     "Investment",
# ]

# df_panel = df_panel.sort_values(["COUNTRY", "YEAR"]).copy()

# for col in volatility_base_cols:
#     if col in df_panel.columns:
#         df_panel[f"{col}_rollstd3"] = (
#             df_panel.groupby("COUNTRY")[col]
#             .transform(
#                 lambda x: x.shift(1).rolling(
#                     window=3,
#                     min_periods=2,
#                 ).std()
#             )
#         )

#         df_panel[f"{col}_rollstd5"] = (
#             df_panel.groupby("COUNTRY")[col]
#             .transform(
#                 lambda x: x.shift(1).rolling(
#                     window=5,
#                     min_periods=3,
#                 ).std()
#             )
#         )

# print("Volatility features added.")

Volatility features added.


In [ ]:
# stationary_cols  = ["GDP_Growth","Inflation","Exports","Imports",
#                     "Fiscal_Balance","Current_Account"]
# differenced_cols = ["Debt_diff","Expenditure_diff","Revenue_diff",
#                     "Savings_diff","Investment_diff"]
# all_feature_cols = stationary_cols + differenced_cols

# for col in all_feature_cols:
#     df_panel[f"{col}_lag1"] = df_panel.groupby("COUNTRY")[col]\
#         .transform(lambda x: x.shift(1))

# # Autoregressive features
# df_panel["GDP_Growth_lag1"]      = df_panel.groupby("COUNTRY")["GDP_Growth"]\
#     .transform(lambda x: x.shift(1))
# df_panel["GDP_Growth_rollmean3"] = df_panel.groupby("COUNTRY")["GDP_Growth"]\
#     .transform(lambda x: x.shift(1).rolling(3, min_periods=2).mean())
# df_panel["Inflation_lag1_log"]   = np.sign(df_panel["Inflation_lag1"]) * \
#     np.log1p(np.abs(df_panel["Inflation_lag1"]))

# lag_feature_cols = [f"{col}_lag1" for col in all_feature_cols]
# print(f"Lag features created: {len(lag_feature_cols)}")
# print(f"Total columns: {df_panel.shape[1]}")

Lag features created: 11
Total columns: 84


In [ ]:
# # Collect newly created feature columns without duplicates
# raw_new_cols = (
#     lag_feature_cols
#     + [
#         col for col in df_panel.columns
#         if "rollmean" in col or "rollstd" in col
#     ]
#     + [
#         "GDP_Growth_lag1",
#         "GDP_Growth_rollmean3",
#         "Inflation_lag1_log",
#     ]
# )

# new_cols = list(dict.fromkeys(raw_new_cols))

# # Lagged and rolling features must not be backward-filled because
# # that would give earlier observations information from future years.
# lag_and_rolling_cols = [
#     col for col in new_cols
#     if "lag" in col or "roll" in col
# ]

# # Keep only rows with a known target and complete predictive history
# df_panel = df_panel.dropna(
#     subset=lag_and_rolling_cols + ["GDP_Growth"]
# )

# print(f"Final shape : {df_panel.shape}")
# print(f"Countries   : {df_panel['COUNTRY'].nunique()}")
# print(
#     "Feature NaN:",
#     df_panel[lag_and_rolling_cols].isna().sum().sum()
# )

Final shape : (4982, 84)
Countries   : 175
Feature NaN: 0


In [14]:
feature_cols = [
    col for col in df_panel.columns
    if "lag" in col or "rollmean" in col or "rollstd" in col
]

df_panel = df_panel.dropna(
    subset=feature_cols + ["GDP_Growth"]
).copy()

df_panel.to_csv(r"data/02_panel_features.csv", index=False)

df_panel[df_panel["YEAR"] <= OBSERVED_END].to_csv(r"data/02_panel_features_observed_1995_2023.csv",
    index=False,
)

df_panel[df_panel["YEAR"].between(SCENARIO_START, SCENARIO_END)].to_csv(
    r"data/02_panel_features_scenario_2024_2026.csv",
    index=False,
)

stationarity_df.to_csv(
    r"data/01_stationarity_training_only.csv",
    index=False,
)

print("Saved:")
print(" - data/02_panel_features.csv")
print(" - data/02_panel_features_observed_1995_2023.csv")
print(" - data/02_panel_features_scenario_2024_2026.csv")
print(" - data/01_stationarity_training_only.csv")
if "Data_Split" not in df_panel.columns:
    df_panel["Data_Split"] = np.select(
        [
            df_panel["YEAR"] <= TRAIN_END,
            df_panel["YEAR"].between(2020, OBSERVED_END),
            df_panel["YEAR"].between(SCENARIO_START, SCENARIO_END),
        ],
        ["train_observed", "test_observed", "scenario_projection"],
        default="outside_scope",
    )

print(df_panel["Data_Split"].value_counts())

Saved:
 - data/02_panel_features.csv
 - data/02_panel_features_observed_1995_2023.csv
 - data/02_panel_features_scenario_2024_2026.csv
 - data/01_stationarity_training_only.csv
Data_Split
train_observed         3952
test_observed           692
scenario_projection     513
Name: count, dtype: int64


In [15]:
# ── Save checkpoint ───────────────────────────────────────────
df_panel.to_csv("data/02_panel_features.csv", index=False)
print("Saved: data/02_panel_features.csv")
print(df_panel.shape)

Saved: data/02_panel_features.csv
(5157, 89)


In [ ]:
df_panel.head()

,COUNTRY,YEAR,Inflation,Current_Account,Expenditure,Exports,Investment,Debt,GDP_Growth,Savings,...,Exports_lag1,Imports_lag1,Fiscal_Balance_lag1,Current_Account_lag1,Debt_diff_lag1,Expenditure_diff_lag1,Revenue_diff_lag1,Savings_diff_lag1,Investment_diff_lag1,Inflation_lag1_log
3,"Afghanistan, Islamic Republic of",2006,6.785,20.844,18.262,-5.943,29.489,22.985,5.361,50.333,...,40.082,54.919,-0.917,30.226,-38.611,0.582,2.057,-5.296,1.694,2.448329
4,"Afghanistan, Islamic Republic of",2007,8.681,63.390,21.448,-12.758,55.852,20.137,13.340,119.243,...,-5.943,-2.315,0.684,20.844,-183.371,2.611,4.213,-16.941,-7.559,2.052199
5,"Afghanistan, Islamic Republic of",2008,26.419,33.769,20.899,-5.885,30.222,19.057,3.863,63.990,...,-12.758,-10.316,-2.462,63.390,-2.848,3.186,0.040,68.910,26.363,2.270165
6,"Afghanistan, Islamic Republic of",2009,-6.811,41.587,21.151,35.076,36.503,16.247,20.585,78.090,...,-5.885,-8.349,-3.864,33.769,-1.080,-0.549,-1.950,-55.253,-25.630,3.311236
7,"Afghanistan, Islamic Republic of",2010,2.179,29.430,20.789,7.125,30.269,7.697,8.438,59.699,...,35.076,30.292,-1.761,41.587,-2.810,0.252,2.355,14.100,6.281,-2.055533
